# Data Skills - Week 11 - Relational databases in Python

**Name:** YOUR NAME

## How to answer

As in Part 1, run each cell in order and read the comments. When a question asks for code, add it to the cell marked `YOUR TURN` (the code is given in the question). When a question asks for an explanation, write it after **Answer:** in the Markdown cell, in your own words.

In Part 1 you used `dbplyr` to write SQL for you. Here you'll write SQL yourself and send it to DuckDB from Python. The results come back as pandas data frames.

## Setup

You need the conda environment from the lab slides (`dataskills-w11`, with `python-duckdb`, `pandas`, `matplotlib`, and `jupyterlab`). This notebook should be in your project's `code/` folder and the four CSV files in `data/`, so the paths below start with `../data/`.

In [ ]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt

# An in-memory database: it disappears when you close the notebook.
# Part 1 saved a database file instead; here we build the tables straight from the CSV files.
con = duckdb.connect()

for table in ["sites", "species", "observers", "observations"]:
    con.sql(f"CREATE OR REPLACE TABLE {table} AS SELECT * FROM read_csv('../data/{table}.csv')")

con.sql("SHOW TABLES")

`CREATE TABLE ... AS SELECT * FROM read_csv(...)` is SQL: DuckDB reads the CSV itself, so the data never pass through pandas. `con.sql()` runs a query and prints a preview. Add `.df()` to get the result as a pandas data frame.

`DESCRIBE` shows each column and the type DuckDB chose for it:

In [ ]:
con.sql("DESCRIBE observations")

## 1. Filter and sort

This query lists the woodpeckers (family Picidae) in the `species` table:

In [ ]:
con.sql("""
    SELECT code, common_name, scientific_name
    FROM species
    WHERE family = 'Picidae'
""")

**Q1.** In the `YOUR TURN` cell, copy the query and change it to list the species in the family Cardinalidae, sorted by common name. Change the `WHERE` line to `WHERE family = 'Cardinalidae'` and add the line `ORDER BY common_name` at the end. How many species are in that family in this table? Note that SQL text values use single quotes and are case-sensitive: what does `WHERE family = 'cardinalidae'` return?

In [ ]:
# YOUR TURN (Q1)

In [ ]:
# YOUR TURN (Q1): try the lowercase version

**Answer:** 

## 2. Summarize by month and plot

This query joins `observations` to `species`, keeps one species (Dickcissel, code `DICK`), and totals the birds counted in each month. `month(date)` pulls the month number out of a date. We use `.df()` to get a pandas data frame we can plot.

In [ ]:
monthly = con.sql("""
    SELECT month(o.date) AS month, SUM(o.count) AS total
    FROM observations o
    JOIN species sp ON o.species_id = sp.species_id
    WHERE sp.code = 'DICK'
    GROUP BY month
    ORDER BY month
""").df()

monthly

In [ ]:
ax = monthly.plot.bar(x="month", y="total", legend=False, color="gray")
ax.set_xlabel("Month")
ax.set_ylabel("Birds counted (2024-2025)")
ax.set_title("Dickcissel counts by month")
plt.show()

**Q2.** In the `YOUR TURN` cell, copy the query and the plot code and change them to Sedge Wren (code `SEWR`); update the title too. In which months were Sedge Wrens counted, and when was the total highest? Some months are missing from the tables (April for Dickcissel; April and May for Sedge Wren). Why? What would you need to do to show those months as zero?

In [ ]:
# YOUR TURN (Q2)

**Answer:** 

## 3. Query a file without loading it

DuckDB can run SQL on a CSV file directly, without creating a table first. It reads only what the query needs, which is how you can work with files that are too big to open in pandas.

In [ ]:
con.sql("""
    SELECT COUNT(*) AS n_rows, MIN(date) AS first_date, MAX(date) AS last_date
    FROM read_csv('../data/observations.csv')
""")

**Q3.** How many rows does `observations.csv` have, and what date range does it cover? Give one situation in your own field where querying a file this way would be better than reading it into a pandas data frame with `pd.read_csv()`.

**Answer:** 

## 4. Write your own query

**Q4.** Write a query that answers: *In 2025, which observer counted the most birds in total at the prairie and farmland sites?* Return one row per observer with the observer's name and the total, largest first. Use this outline:

- `SELECT` the observer name and `SUM(o.count) AS total`
- `FROM observations o`, joined to `sites s` (on `site_id`) and to `observers ob` (on `observer_id`)
- `WHERE s.habitat IN ('prairie', 'farmland') AND o.date >= '2025-01-01'`
- `GROUP BY` the observer name, then `ORDER BY total DESC`

Who counted the most birds, and what was the total? Why do you need two joins for this question?

**Hints:** See the *SQL queries – joins* and *Notes on SQL syntax* slides from lecture (clause order matters). The Q2 query shows the join syntax.

In [ ]:
# YOUR TURN (Q4)

**Answer:** 

In [ ]:
# Close the connection when you're done
con.close()

## Submission

1. Fill in your name at the top.
2. Restart the kernel and run all cells (*Kernel > Restart Kernel and Run All Cells*), then export to HTML (*File > Save and Export Notebook As > HTML*). Don't check "Sanitize HTML".
3. Commit and push this notebook and the HTML to your `dataskills-w11-LASTNAME` repository.
4. Upload this `.ipynb` file and the HTML file to Canvas.